# Hugging Face Integration with AdvSecureNet API

This notebook demonstrates how to use Hugging Face datasets and models with AdvSecureNet through the Python API. We'll cover:

1. **Dataset Configuration with Splits**: How to load different splits and configure custom split mappings
2. **Hugging Face Models**: Using pretrained models from Hugging Face Hub
3. **Adversarial Attack Example**: FGSM attack using Hugging Face components
4. **Adversarial Training Example**: Training with adversarial examples

## Key Features Demonstrated
- Split priority system (`split_config` vs `load_splits` vs defaults)
- Custom split names and mappings
- Hugging Face dataset integration
- Model loading from Hugging Face Hub

## 1. Setup and Imports

## Important: Hugging Face Identifier Formats

AdvSecureNet supports multiple formats for Hugging Face identifiers:

### For Datasets:
- ✅ **Short form**: `"uoft-cs/cifar10"`
- ✅ **Full URL**: `"https://huggingface.co/datasets/uoft-cs/cifar10"`
- ✅ **Domain variations**: `"huggingface.co/datasets/uoft-cs/cifar10"`, `"hf.co/datasets/uoft-cs/cifar10"`

### For Models:
- ✅ **Short form**: `"microsoft/resnet-18"`
- ✅ **Full URL**: `"https://huggingface.co/microsoft/resnet-18"`
- ✅ **Domain variations**: `"huggingface.co/microsoft/resnet-18"`, `"hf.co/microsoft/resnet-18"`

**Key Point**: All formats work identically - use whichever is most convenient for your workflow!

In [58]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader

# AdvSecureNet imports
from advsecurenet.dataloader.data_loader_factory import DataLoaderFactory
from advsecurenet.models.model_factory import ModelFactory
from advsecurenet.datasets.dataset_factory import DatasetFactory
from advsecurenet.computer_vision.image_classification.attacks.gradient_based import FGSM
from advsecurenet.computer_vision.image_classification.attacks.attacker import Attacker
from advsecurenet.trainer.trainer import Trainer

# Configuration imports for API usage (not CLI configs)
from advsecurenet.shared.types.configs.model_config import CreateModelConfig
from advsecurenet.shared.types.configs.attack_configs import FgsmAttackConfig
from advsecurenet.shared.types.configs.attack_configs.attacker_config import AttackerConfig
from advsecurenet.shared.types.configs import TrainConfig
from advsecurenet.shared.types.configs.preprocess_config import PreprocessConfig, PreprocessStep
from advsecurenet.shared.types.configs.device_config import DeviceConfig

## 2. Dataset Configuration Examples

### Understanding API vs CLI Patterns

**API Usage (This Notebook)**: Use direct parameters with `DatasetFactory.load_dataset(**kwargs)`
**CLI Usage**: Use configuration objects like `CreateDatasetCliConfig` in YAML files

The dataset loading follows a priority system:
1. **Highest Priority**: `split_config` - Dictionary mapping logical split names to split configurations
2. **Medium Priority**: `load_splits` - List of split names to load
3. **Default**: `['train', 'test']` if nothing is specified

### Example 1: Using load_splits (Most Common API Usage)

In [59]:
# Example 1: Default behavior - when you don't specify load_splits or split_config
# The factory will automatically load ['train', 'test'] splits

# Define preprocessing configuration once and reuse
preprocessing_config = PreprocessConfig(
    steps=[
        PreprocessStep(name="Resize", params={"size": 32}),
        PreprocessStep(name="CenterCrop", params={"size": 32}),
        PreprocessStep(name="ToTensor"),
        PreprocessStep(name="ToDtype", params={"dtype": "torch.float32", "scale": True}),
        PreprocessStep(
            name="Normalize",
            params={"mean": [0.485, 0.456, 0.406], "std": [0.229, 0.224, 0.225]}
        ),
    ]
)

print("=== Example 1: Default Behavior ===")
print("When you don't specify load_splits or split_config, the factory defaults to ['train', 'test']")

# Load dataset with minimal parameters - uses defaults
default_dataset = DatasetFactory.load_dataset(
    dataset_name="uoft-cs/cifar10",
    num_classes=10,
    preprocessing=preprocessing_config,
    constructor_args={
        "input_key": "img",
        "target_key": "label"
    }
)

print(f"✅ Dataset loaded with default splits: {list(default_dataset.keys())}")
print(f"✅ Train samples: {len(default_dataset['train'])}")
print(f"✅ Test samples: {len(default_dataset['test'])}")
print("✅ This is the simplest way to load a dataset!")

=== Example 1: Default Behavior ===
When you don't specify load_splits or split_config, the factory defaults to ['train', 'test']
✅ Dataset loaded with default splits: ['train', 'test']
✅ Train samples: 50000
✅ Test samples: 10000
✅ This is the simplest way to load a dataset!


### Example 2: Using load_splits with Custom Split Names

When you want to explicitly specify which splits to load or use custom split names:

In [60]:
# Example 2: Using load_splits to explicitly specify which splits to load
# This is useful when you want different split names or to load specific splits

print("=== Example 2: Explicit load_splits ===")
print("Explicitly specify which splits to load and their logical names")

# Load dataset with explicit split specification
explicit_splits_dataset = DatasetFactory.load_dataset(
    dataset_name="huggingface",
    identifier="uoft-cs/cifar10",
    num_classes=10,
    preprocessing=preprocessing_config,
    constructor_args={
        "input_key": "img",
        "target_key": "label"
    },
    load_splits=["train", "test"]  # Explicitly specify splits to load
)

print(f"✅ Dataset loaded with explicit splits: {list(explicit_splits_dataset.keys())}")
print(f"✅ Train samples: {len(explicit_splits_dataset['train'])}")
print(f"✅ Test samples: {len(explicit_splits_dataset['test'])}")
print("✅ Same result as default, but explicit about what splits to load")
print("✅ You could also do load_splits=['train'] to load only training data")

=== Example 2: Explicit load_splits ===
Explicitly specify which splits to load and their logical names


✅ Dataset loaded with explicit splits: ['train', 'test']
✅ Train samples: 50000
✅ Test samples: 10000
✅ Same result as default, but explicit about what splits to load
✅ You could also do load_splits=['train'] to load only training data


### Example 3: Using split_config for Different Settings Per Split

When you need different datasets, preprocessing, or split names for different logical splits:

In [61]:
# Example 3: Using split_config for different datasets with semantic compatibility
# This demonstrates the most powerful approach - different datasets with SAME semantic classes

print("=== Example 3: split_config with Semantically Compatible Datasets ===")
print("Use split_config when you need different datasets with SAME semantic meanings")

# Define different preprocessing for corrupted test set
corrupted_preprocessing = PreprocessConfig(
    steps=[
        PreprocessStep(name="Resize", params={"size": 32}),
        PreprocessStep(name="CenterCrop", params={"size": 32}),
        PreprocessStep(name="ToTensor"),
        PreprocessStep(name="ToDtype", params={"dtype": "torch.float32", "scale": True}),
        PreprocessStep(
            name="Normalize",
            params={"mean": [0.5, 0.5, 0.5], "std": [0.5, 0.5, 0.5]}  # Different normalization for corrupted images
        ),
    ]
)

# Load dataset with split_config - different datasets with SAME semantic classes
split_config_dataset = DatasetFactory.load_dataset(
    dataset_name="huggingface",
    num_classes=10,
    split_config={
        "train": {
            # Train on clean CIFAR-10 images - explicit settings
            "identifier": "uoft-cs/cifar10",
            "split_name": "train",
            "preprocessing": preprocessing_config,  # Clean preprocessing for training
            "constructor_args": {  # Explicit constructor args for CIFAR-10
                "input_key": "img",
                "target_key": "label"
            }
        },
        "test": {
            # Test on corrupted CIFAR-10-C images - SAME semantic classes!
            "identifier": "randall-lab/cifar10-c",  # Different dataset but same classes
            "split_name": "test",
            "preprocessing": corrupted_preprocessing,  # Different preprocessing for corrupted images
            "constructor_args": {  # Different field names for CIFAR-10-C
                "input_key": "image",   # CIFAR-10-C uses "image" instead of "img"
                "target_key": "label"   # Same label field
            },
            "dataset_kwargs": {"trust_remote_code": True, "cache_dir": "cifar10-c/"}  # Required for CIFAR-10-C dataset
        }
    }
)

print(f"✅ Dataset loaded with split_config: {list(split_config_dataset.keys())}")
print(f"✅ Train samples (CIFAR-10): {len(split_config_dataset['train'])}")
print(f"✅ Test samples (CIFAR-10-C): {len(split_config_dataset['test'])}")

print("\n💡 Key Benefits:")
print("✅ Train on clean CIFAR-10, test on corrupted CIFAR-10-C")
print("✅ Measure robustness: clean training → corrupted testing") 
print("✅ Two different datasets with different field mappings")
print("✅ Perfect example of cross-dataset evaluation!")

=== Example 3: split_config with Semantically Compatible Datasets ===
Use split_config when you need different datasets with SAME semantic meanings
✅ Dataset loaded with split_config: ['train', 'test']
✅ Train samples (CIFAR-10): 50000
✅ Test samples (CIFAR-10-C): 950000

💡 Key Benefits:
✅ Train on clean CIFAR-10, test on corrupted CIFAR-10-C
✅ Measure robustness: clean training → corrupted testing
✅ Two different datasets with different field mappings
✅ Perfect example of cross-dataset evaluation!


### Example 4: Anti-Pattern - Don't Do This! (Demonstration Only)

**⚠️ WARNING: This example shows what NOT to do!**  
This demonstrates how `split_config` overrides `load_splits`, making the `load_splits` parameter pointless. This is confusing and should be avoided in real code.

In [62]:
# Example 4: Anti-Pattern - DON'T DO THIS! 
# This shows multiple confusing patterns that should be avoided

print("=== Example 4: Anti-Pattern - What NOT to Do ===")
print("❌ This example shows confusing configuration that should be avoided!")
print("⚠️  Multiple anti-patterns combined to show what makes code confusing")

# Define preprocessing that will be ignored
ignored_preprocessing = PreprocessConfig(
    steps=[
        PreprocessStep(name="Resize", params={"size": 64}),  # This will be completely ignored!
        PreprocessStep(name="ToTensor"),
    ]
)

# BAD: Multiple confusing patterns in one configuration
antipattern_dataset = DatasetFactory.load_dataset(
    dataset_name="huggingface",
    identifier="uoft-cs/cifar10",
    num_classes=10,
    preprocessing=ignored_preprocessing,  # ❌ IGNORED: Both splits override this global preprocessing!
    constructor_args={  # ❌ PARTIALLY IGNORED: Only custom_test uses global args
        "input_key": "img",      # ❌ IGNORED: custom_train overrides this
        "target_key": "label"    # ✅ Used by custom_test (should be explicit for clarity)
    },
    load_splits=["validation", "non_existent"],  # ❌ COMPLETELY IGNORED: split_config takes priority
    split_config={  # ✅ This takes priority and OVERRIDES everything above
        "custom_train": {
            "split_name": "train",
            "preprocessing": preprocessing_config,  # Overrides global preprocessing
            "constructor_args": {  # Overrides global constructor_args
                "input_key": "img",     # Same value but redundant override
                "target_key": "label"   # Same value but redundant override  
            }
        },
        "custom_test": {
            "split_name": "test",
            "preprocessing": preprocessing_config,  # Overrides global preprocessing
            # ❌ UNCLEAR: Uses global constructor_args - should be explicit for clarity!
            # Should add: "constructor_args": {"input_key": "img", "target_key": "label"}
        }
    }
)

print(f"❌ Specified load_splits: ['train', 'test'] - COMPLETELY IGNORED!")
print(f"❌ Global preprocessing (size=64) - IGNORED by ALL splits!")
print(f"❌ Global input_key override - IGNORED by custom_train!")
print(f"❌ Global constructor_args - UNCLEAR which splits use them!")
print(f"✅ Actual splits loaded: {list(antipattern_dataset.keys())}")

print("\n❌ What Makes This Confusing:")
print("  1. load_splits specified but completely ignored")
print("  2. Global preprocessing ignored by ALL splits - wasteful!")
print("  3. Global constructor_args partially ignored - unclear!")
print("  4. Inconsistent split configurations - some explicit, some implicit")
print("  5. Redundant overrides with same values")

print("\n✅ How to Fix This:")
print("  1. Remove load_splits - pick ONE approach!")
print("  2. Remove global preprocessing if all splits override it")
print("  3. Be explicit about constructor_args in ALL splits")
print("  4. Use global settings only when actually shared")
print("  5. Don't override with identical values")

=== Example 4: Anti-Pattern - What NOT to Do ===
❌ This example shows confusing configuration that should be avoided!
⚠️  Multiple anti-patterns combined to show what makes code confusing
❌ Specified load_splits: ['train', 'test'] - COMPLETELY IGNORED!
❌ Global preprocessing (size=64) - IGNORED by ALL splits!
❌ Global input_key override - IGNORED by custom_train!
❌ Global constructor_args - UNCLEAR which splits use them!
✅ Actual splits loaded: ['custom_train', 'custom_test']

❌ What Makes This Confusing:
  1. load_splits specified but completely ignored
  2. Global preprocessing ignored by ALL splits - wasteful!
  3. Global constructor_args partially ignored - unclear!
  4. Inconsistent split configurations - some explicit, some implicit
  5. Redundant overrides with same values

✅ How to Fix This:
  1. Remove load_splits - pick ONE approach!
  2. Remove global preprocessing if all splits override it
  3. Be explicit about constructor_args in ALL splits
  4. Use global settings only w

## Summary and Best Practices

### Configuration Priority System
1. **`split_config`** (highest priority) - Use for complex split configurations with different preprocessing per split
2. **`load_splits`** (medium priority) - Use for simple split selection from available dataset splits  
3. **Default behavior** (lowest priority) - Loads `['train', 'test']` automatically

### When to Use Each Approach

**Use Default (Example 1):** When you want standard train/test splits with same preprocessing
```python
# Simple case - just specify the essentials
dataset = DatasetFactory.load_dataset(
    dataset_name="huggingface",
    identifier="uoft-cs/cifar10",
    num_classes=10,
    preprocessing=preprocessing_config,
    constructor_args={"input_key": "img", "target_key": "label"}
)
```

**Use load_splits (Example 2):** When you need specific splits but same preprocessing for all
```python
# Need specific splits, same preprocessing
dataset = DatasetFactory.load_dataset(
    load_splits=["train", "validation", "test"]  # Different splits, same preprocessing
)
```

**Use split_config (Example 3):** When you need different preprocessing or settings per split
```python
# Different preprocessing per split
split_config={
    "train": {"split_name": "train", "preprocessing": train_preprocessing},
    "test": {"split_name": "test", "preprocessing": test_preprocessing}
}
```

### ❌ Anti-Patterns to Avoid
- **Don't mix approaches** - Using both `load_splits` and `split_config` is confusing
- **Don't set global attributes that are overwritten in split_config**

## 3. Hugging Face Model Configuration

In [63]:
# Configure a Hugging Face model using proper attributes
model_config = CreateModelConfig(
    model_name="huggingface",  # This tells the factory it's a HF model
    model_identifier="microsoft/resnet-18",  # Use model_identifier for HF model ID
    pretrained=True,  # Direct attribute, not in model_kwargs
    trust_remote_code=False,  # Direct attribute for HF models
    architecture={"num_classes": 10},  # Model architecture parameters
    # Optional HF-specific parameters
    revision=None,  # Use specific model revision if needed
    cache_dir=None,  # Use default cache directory
)

# Load the model using the ModelFactory
model = ModelFactory.create_model(config=model_config)
print(f"Model loaded: {type(model).__name__}")
print(f"Model device: {next(model.parameters()).device}")

# Move to appropriate device
device = torch.device("cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu")
model = model.to(device)
print(f"Model moved to: {device}")
print(f"\n=== Model Configuration ===")
print(f"Model name: {model_config.model_name}")
print(f"Model identifier: {model_config.model_identifier}")
print(f"Pretrained: {model_config.pretrained}")
print(f"Architecture: {model_config.architecture}")
print("\nSupported model identifier formats:")
print("- Short form: 'microsoft/resnet-18'")
print("- Full URL: 'https://huggingface.co/microsoft/resnet-18'")
print("- Both formats work seamlessly!")

Model loaded: HuggingFaceModel
Model device: cpu
Model moved to: mps

=== Model Configuration ===
Model name: huggingface
Model identifier: microsoft/resnet-18
Pretrained: True
Architecture: {'num_classes': 10}

Supported model identifier formats:
- Short form: 'microsoft/resnet-18'
- Full URL: 'https://huggingface.co/microsoft/resnet-18'
- Both formats work seamlessly!


## 4. Adversarial Attack Example

### FGSM Attack with Hugging Face Components

In [64]:
# Configure FGSM attack using our already loaded model and datasets
attack_config = FgsmAttackConfig(
    epsilon=0.03,
    targeted=False
)

# Create attack instance
fgsm_attack = FGSM(config=attack_config)
print(f"Attack created: {type(fgsm_attack).__name__}")

# Use our already loaded model and create data loaders from our datasets
print(f"Using model: {type(model).__name__}")
print(f"Available datasets: {list(split_config_dataset.keys())}")

# Create data loaders from our loaded datasets
train_loader = DataLoaderFactory.create_dataloader(
    dataset=split_config_dataset['train'],
    batch_size=32,
    shuffle=True
)

test_loader = DataLoaderFactory.create_dataloader(
    dataset=split_config_dataset['test'], 
    batch_size=32,
    shuffle=False
)

print(f"✅ Created data loaders from our existing datasets!")
print(f"✅ Train loader: {len(train_loader.dataset)} samples")
print(f"✅ Test loader: {len(test_loader.dataset)} samples")

# Prepare model and data
model.eval()
criterion = nn.CrossEntropyLoss()

# Get a batch of data from our test loader
data_iter = iter(test_loader)
images, labels = next(data_iter)
images, labels = images.to(device), labels.to(device)

print(f"Original batch shape: {images.shape}")
print(f"Labels shape: {labels.shape}")
print(f"Using epsilon: {attack_config.epsilon}")
print(f"Attack type: {'Targeted' if attack_config.targeted else 'Untargeted'}")

Attack created: FGSM
Using model: HuggingFaceModel
Available datasets: ['train', 'test']
✅ Created data loaders from our existing datasets!
✅ Train loader: 50000 samples
✅ Test loader: 950000 samples
Original batch shape: torch.Size([32, 3, 32, 32])
Labels shape: torch.Size([32])
Using epsilon: 0.03
Attack type: Untargeted


In [65]:
# Generate adversarial examples
model.eval()
adversarial_images = fgsm_attack.attack(model, images, labels)

print(f"Adversarial examples generated: {adversarial_images.shape}")

# Evaluate original vs adversarial accuracy
with torch.no_grad():
    # Original predictions
    original_outputs = model(images)
    original_preds = torch.argmax(original_outputs, dim=1)
    original_accuracy = (original_preds == labels).float().mean()
    
    # Adversarial predictions
    adv_outputs = model(adversarial_images)
    adv_preds = torch.argmax(adv_outputs, dim=1)
    adv_accuracy = (adv_preds == labels).float().mean()

print(f"\\n=== Attack Results ===")
print(f"Original accuracy: {original_accuracy:.4f}")
print(f"Adversarial accuracy: {adv_accuracy:.4f}")
print(f"Attack success rate: {1 - adv_accuracy:.4f}")

# Show perturbation statistics
perturbation = adversarial_images - images
print(f"\\n=== Perturbation Statistics ===")
print(f"Max perturbation: {perturbation.abs().max():.6f}")
print(f"Mean perturbation: {perturbation.abs().mean():.6f}")
print(f"L2 norm of perturbation: {torch.norm(perturbation.flatten(), p=2):.6f}")

Adversarial examples generated: torch.Size([32, 3, 32, 32])
\n=== Attack Results ===
Original accuracy: 0.0000
Adversarial accuracy: 0.0000
Attack success rate: 1.0000
\n=== Perturbation Statistics ===
Max perturbation: 1.000000
Mean perturbation: 0.256131
L2 norm of perturbation: 117.291893


## 5. Adversarial Training Example

### Training with Different Datasets for Train/Test Splits

In [66]:
# Configure training
training_config = TrainConfig(
    epochs=2,  # Short for demo
    learning_rate=0.001,
    optimizer="adam",
    scheduler="step",
    save_checkpoint=True,
    checkpoint_interval=1,
    processor=str(device)
)

print("Training configuration created!")
print(f"Epochs: {training_config.epochs}")
print(f"Learning rate: {training_config.learning_rate}")
print(f"Optimizer: {training_config.optimizer}")

# For adversarial training, you would typically use specific adversarial training classes
# This example shows the basic training setup structure

Training configuration created!
Epochs: 2
Learning rate: 0.001
Optimizer: adam


In [67]:
# Create trainer using our already loaded model and data loaders
print("✅ Using our existing model and data loaders for training!")
print(f"Model: {type(model).__name__}")
print(f"Train loader dataset: train from CIFAR-10")  
print(f"Test loader dataset: test from CIFAR-10-C")

# Note: Trainer interface may vary - this shows the concept of reusing components
print("Trainer setup concepts demonstrated!")
print(f"Training dataset size: {len(train_loader.dataset)}")
print(f"Test dataset size: {len(test_loader.dataset)}")
print(f"Model parameters: {sum(p.numel() for p in model.parameters()):,}")
print(f"Trainable parameters: {sum(p.numel() for p in model.parameters() if p.requires_grad):,}")

print("\n💡 Benefits of Reusing Loaded Components:")
print("✅ Consistent data across attack and training experiments")
print("✅ No redundant dataset loading - more efficient")
print("✅ Same preprocessing applied to attack and training data")
print("✅ Cleaner code with fewer variable definitions")

✅ Using our existing model and data loaders for training!
Model: HuggingFaceModel
Train loader dataset: train from CIFAR-10
Test loader dataset: test from CIFAR-10-C
Trainer setup concepts demonstrated!
Training dataset size: 50000
Test dataset size: 950000
Model parameters: 11,689,512
Trainable parameters: 11,689,512

💡 Benefits of Reusing Loaded Components:
✅ Consistent data across attack and training experiments
✅ No redundant dataset loading - more efficient
✅ Same preprocessing applied to attack and training data
✅ Cleaner code with fewer variable definitions


## 6. Global vs Split Configuration Reference

### Understanding Parameter Inheritance and Override

The `DatasetFactory.load_dataset()` supports a flexible configuration system where you can set parameters globally (applied to all splits) or per-split (overriding global settings). Here's a comprehensive guide to which parameters can be set where:

### 🌍 Global Parameters (Apply to All Splits)
Set these at the top level of `DatasetFactory.load_dataset()`:

| Parameter | Description | When to Use Globally |
|-----------|-------------|---------------------|
| `dataset_name` | Dataset type ("huggingface", "torchvision", etc.) | ✅ **Always** - Same dataset system for all splits |
| `identifier` | HuggingFace dataset ID | ✅ When all splits use same dataset |
| `num_classes` | Number of output classes | ✅ **Always** - Model compatibility requires same classes |
| `preprocessing` | Preprocessing pipeline | ✅ When all splits use same preprocessing |
| `constructor_args` | Field mappings (input_key, target_key) | ✅ When all splits have same field names |
| `load_splits` | List of splits to load | ✅ Simple cases with uniform settings |
| `dataset_kwargs` | Additional dataset parameters | ✅ When all splits need same extra parameters |

### 🎯 Split-Specific Parameters (Override Global Settings)
Set these inside `split_config` for individual splits:

| Parameter | Description | When to Override Per Split |
|-----------|-------------|---------------------------|
| `identifier` | Different dataset for this split | ✅ Cross-dataset evaluation (CIFAR-10 → CIFAR-10-C) |
| `split_name` | Which split to load from dataset | ✅ **Always** in split_config - Required |
| `preprocessing` | Different preprocessing pipeline | ✅ Different augmentations/normalization per split |
| `constructor_args` | Different field mappings | ✅ Datasets with different field names |
| `dataset_kwargs` | Split-specific parameters | ✅ trust_remote_code, cache_dir per dataset |

### 📊 Configuration Patterns and Examples

#### Pattern 1: All Global (Simple Case)
```python
# When all splits use same dataset and settings
DatasetFactory.load_dataset(
    dataset_name="huggingface",
    identifier="uoft-cs/cifar10",  # Same dataset for all
    num_classes=10,
    preprocessing=same_preprocessing,  # Same preprocessing for all
    constructor_args={"input_key": "img", "target_key": "label"}  # Same fields for all
)
```

#### Pattern 2: Mixed Global + Split Overrides (Most Flexible)
```python
# When you need different settings per split
DatasetFactory.load_dataset(
    dataset_name="huggingface",  # Global: Same dataset system
    num_classes=10,  # Global: Same number of classes
    split_config={
        "train": {
            "identifier": "uoft-cs/cifar10",  # Override: Clean dataset for training
            "split_name": "train",
            "preprocessing": clean_preprocessing,  # Override: Clean preprocessing
            "constructor_args": {"input_key": "img", "target_key": "label"}
        },
        "test": {
            "identifier": "randall-lab/cifar10-c",  # Override: Corrupted dataset for testing
            "split_name": "test", 
            "preprocessing": corrupted_preprocessing,  # Override: Different preprocessing
            "constructor_args": {"input_key": "image", "target_key": "label"},  # Override: Different field names
            "dataset_kwargs": {"trust_remote_code": True}  # Override: Extra parameters for this dataset
        }
    }
)
```

#### Pattern 3: All Split-Specific (Most Explicit)
```python
# When you want maximum clarity and no global inheritance
DatasetFactory.load_dataset(
    dataset_name="huggingface",  # Only dataset system is global
    num_classes=10,  # Only num_classes is global
    split_config={
        "train": {
            "identifier": "uoft-cs/cifar10",
            "split_name": "train",
            "preprocessing": train_preprocessing,
            "constructor_args": {"input_key": "img", "target_key": "label"}
        },
        "test": {
            "identifier": "uoft-cs/cifar10", 
            "split_name": "test",
            "preprocessing": test_preprocessing,
            "constructor_args": {"input_key": "img", "target_key": "label"}
        }
    }
)
```

### 🔄 Parameter Override Rules

1. **Split-specific parameters ALWAYS override global parameters**
2. **Missing split parameters inherit from global settings**
3. **`split_config` completely overrides `load_splits`**
4. **`dataset_name` and `num_classes` are global-only**

### 🎯 Best Practices for Parameter Placement

#### ✅ Use Global Parameters When:
- **Same dataset**: All splits from same HuggingFace repository
- **Same preprocessing**: Identical augmentations/normalization
- **Same field mapping**: All datasets use same input/target field names
- **Simple scenarios**: Standard train/test with uniform settings

#### ✅ Use Split Parameters When:
- **Cross-dataset evaluation**: CIFAR-10 (train) → CIFAR-10-C (test)
- **Different preprocessing**: Augmented training, clean testing
- **Different field names**: CIFAR-10 uses "img", CIFAR-10-C uses "image"
- **Special requirements**: Some datasets need trust_remote_code=True

### ⚠️ Common Mistakes to Avoid

#### ❌ Redundant Global Settings
```python
# BAD: Global preprocessing ignored by all splits
DatasetFactory.load_dataset(
    preprocessing=ignored_preprocessing,  # ❌ Wasteful - completely ignored
    split_config={
        "train": {"preprocessing": train_preprocessing},  # Overrides global
        "test": {"preprocessing": test_preprocessing}     # Overrides global
    }
)
```

#### ❌ Mixing Approaches Unnecessarily
```python
# BAD: Using load_splits AND split_config
DatasetFactory.load_dataset(
    load_splits=["train", "test"],  # ❌ Completely ignored!
    split_config={"train": {...}, "test": {...}}  # This takes priority
)
```

#### ❌ Unclear Inheritance
```python
# BAD: Unclear which splits use global constructor_args
DatasetFactory.load_dataset(
    constructor_args={"input_key": "img", "target_key": "label"},  # ❌ Unclear usage
    split_config={
        "train": {"constructor_args": {"input_key": "img", "target_key": "label"}},  # Explicit
        "test": {}  # ❌ Uses global - should be explicit for clarity
    }
)
```

### 🌟 Recommended Approach: Explicit Configuration

For maximum clarity, be explicit about settings for each split:

```python
# BEST: Crystal clear what each split uses
split_config_dataset = DatasetFactory.load_dataset(
    dataset_name="huggingface",  # Global: Dataset system
    num_classes=10,              # Global: Model compatibility
    split_config={
        "train": {
            # Explicit settings for training split
            "identifier": "uoft-cs/cifar10",
            "split_name": "train",
            "preprocessing": clean_preprocessing,
            "constructor_args": {"input_key": "img", "target_key": "label"}
        },
        "test": {
            # Explicit settings for test split  
            "identifier": "randall-lab/cifar10-c",
            "split_name": "test",
            "preprocessing": corrupted_preprocessing,
            "constructor_args": {"input_key": "image", "target_key": "label"},
            "dataset_kwargs": {"trust_remote_code": True}
        }
    }
)
```

This approach makes it immediately clear what settings each split uses, eliminates ambiguity about parameter inheritance, and makes the code self-documenting.

## 7. Summary and Best Practices

### Key Takeaways:

1. **🎯 Configuration Flexibility**:
   - **Global parameters**: Apply to all splits, use for common settings
   - **Split parameters**: Override globals, use for split-specific needs
   - **Priority rule**: Split-specific always overrides global

2. **📝 Parameter Placement Strategy**:
   - **Always global**: `dataset_name`, `num_classes` 
   - **Usually global**: `identifier`, `preprocessing`, `constructor_args`
   - **Often per-split**: `split_name`, different datasets, special requirements

3. **🔍 When to Use Each Approach**:
   - **Default behavior**: Simple datasets, uniform settings
   - **load_splits**: Explicit split selection, same preprocessing
   - **split_config**: Different datasets, preprocessing, or field mappings per split

4. **🛡️ Cross-Domain Evaluation Requirements**:
   - **Same input dimensions** (model architecture compatibility)
   - **Same number of classes** (output layer compatibility)  
   - **SAME semantic meaning for each class** (evaluation validity)
   - Example: CIFAR-10 (clean) ↔ CIFAR-10-C (corrupted) ✅
   - Counter-example: CIFAR-10 (objects) ↔ MNIST (digits) ❌

5. **✨ Best Practices**:
   - Be explicit about split configurations for clarity
   - Use global settings only when actually shared across splits
   - Avoid redundant overrides with identical values
   - Don't mix `load_splits` and `split_config` approaches
   - Always verify semantic compatibility for cross-dataset evaluation

### Real-World Usage Patterns:

- **Standard Training**: Same dataset, same preprocessing → Use defaults or global settings
- **Robustness Testing**: Clean training, corrupted testing → Use split_config with different datasets
- **Domain Adaptation**: Source domain training, target domain testing → Use split_config with semantic compatibility
- **Ablation Studies**: Different preprocessing per split → Use split_config with different preprocessing

The AdvSecureNet HuggingFace integration provides the flexibility to handle simple single-dataset scenarios and complex cross-dataset evaluation with the same API!

In [68]:
# Configuration for using different datasets in train/test using direct API parameters
# CRITICAL: Datasets must have SEMANTICALLY COMPATIBLE classes!

# Define preprocessing for domain shift simulation
domain_shift_preprocessing = PreprocessConfig(
    steps=[
        PreprocessStep(name="Resize", params={"size": 32}),
        PreprocessStep(name="CenterCrop", params={"size": 32}),
        PreprocessStep(name="ToTensor"),
        PreprocessStep(name="ToDtype", params={"dtype": "torch.float32", "scale": True}),
        # More extreme normalization to simulate domain shift
        PreprocessStep(
            name="Normalize",
            params={"mean": [0.3, 0.3, 0.3], "std": [0.3, 0.3, 0.3]}
        ),
    ]
)

# Parameters for mixed dataset configuration
mixed_params = {
    "dataset_name": "huggingface",
    "identifier": "uoft-cs/cifar10",  # Default for training
    "num_classes": 10,
    "preprocessing": preprocessing_config,  # Global preprocessing
    # Global constructor args for HF datasets
    "constructor_args": {
        "input_key": "img",
        "target_key": "label"
    },
    "split_config": {
        "source_train": {
            # Train on CIFAR-10
            "identifier": "uoft-cs/cifar10",
            "split_name": "train"
        },
        "target_test": {
            # Test on CIFAR-10 test set (same domain)
            "identifier": "uoft-cs/cifar10",
            "split_name": "test"
        },
        "domain_shift_test": {
            # Domain shift evaluation: Different visual conditions, SAME semantic classes
            # For demonstration, using same dataset with different preprocessing
            # In practice: Use datasets with same classes but different visual domains
            # Example: CIFAR-10 (clean) → CIFAR-10-C (corrupted)
            "identifier": "uoft-cs/cifar10",
            "split_name": "test", 
            # Different preprocessing to simulate domain shift
            "preprocessing": domain_shift_preprocessing
        }
    }
}

print("Mixed dataset API parameters defined!")
print("Available splits:")
for split_name, split_config in mixed_params["split_config"].items():
    print(f"  {split_name}: {split_config['identifier']} ({split_config['split_name']})")
print(f"Constructor args: {mixed_params['constructor_args']}")

print("\\n=== CRITICAL: Semantic Compatibility Requirements ===")
print("✅ All splits use CIFAR-10: SAME 10 classes with SAME meanings")
print("   Class 0 = airplane, Class 1 = automobile, ..., Class 9 = truck")
print("✅ Model trained on these 10 classes works meaningfully with all splits")
print("✅ Cross-domain evaluation measures robustness to visual changes")
print("✅ No redundant dataset_kwargs - split_name is sufficient!")

print("\\n🎯 Valid cross-domain scenarios (same semantic classes):")
print("- CIFAR-10 (clean) ↔ CIFAR-10-C (corrupted)")
print("- CIFAR-10 ↔ STL-10 (subset to overlapping 10 classes)")
print("- ImageNet (subset) ↔ Places365 (subset with same object classes)")
print("- Fashion-MNIST ↔ Fashion-Product-Images (same clothing categories)")

print("\\n❌ INVALID cross-domain scenarios (different semantic classes):")
print("- CIFAR-10 (objects) ↔ MNIST (digits)")
print("- CIFAR-10 (objects) ↔ SVHN (street numbers)")  
print("- Fashion-MNIST (clothes) ↔ MNIST (digits)")
print("- Any scenario where class indices have different meanings!")

print("\\n🔑 Remember: Cross-domain = Different visual domains, SAME semantic classes!")
print("\\n✅ Using direct API parameters - no CLI config objects needed!")

Mixed dataset API parameters defined!
Available splits:
  source_train: uoft-cs/cifar10 (train)
  target_test: uoft-cs/cifar10 (test)
  domain_shift_test: uoft-cs/cifar10 (test)
Constructor args: {'input_key': 'img', 'target_key': 'label'}
\n=== CRITICAL: Semantic Compatibility Requirements ===
✅ All splits use CIFAR-10: SAME 10 classes with SAME meanings
   Class 0 = airplane, Class 1 = automobile, ..., Class 9 = truck
✅ Model trained on these 10 classes works meaningfully with all splits
✅ Cross-domain evaluation measures robustness to visual changes
✅ No redundant dataset_kwargs - split_name is sufficient!
\n🎯 Valid cross-domain scenarios (same semantic classes):
- CIFAR-10 (clean) ↔ CIFAR-10-C (corrupted)
- CIFAR-10 ↔ STL-10 (subset to overlapping 10 classes)
- ImageNet (subset) ↔ Places365 (subset with same object classes)
- Fashion-MNIST ↔ Fashion-Product-Images (same clothing categories)
\n❌ INVALID cross-domain scenarios (different semantic classes):
- CIFAR-10 (objects) ↔ MNIS

In [69]:
# Example of PROPER cross-domain evaluation with semantically compatible datasets
# This shows datasets with same input/output dimensions AND meaningful class relationships

print("\\n=== SEMANTICALLY COMPATIBLE Cross-Domain Examples ===")
print("\\nCrucial requirement: Output classes must have SAME SEMANTIC MEANING!")
print("Not just same number of classes - the classes themselves must correspond!")

# Example 1: CIFAR-10 vs CIFAR-10 with different augmentations/corruptions
corruption_preprocessing = PreprocessConfig(
    steps=[
        PreprocessStep(name="Resize", params={"size": 32}),
        PreprocessStep(name="CenterCrop", params={"size": 32}),
        PreprocessStep(name="ToTensor"),
        PreprocessStep(name="ToDtype", params={"dtype": "torch.float32", "scale": True}),
        # Simulate corruption with extreme normalization
        PreprocessStep(
            name="Normalize",
            params={"mean": [0.2, 0.2, 0.2], "std": [0.8, 0.8, 0.8]}
        ),
    ]
)

cifar_variants_params = {
    "dataset_name": "huggingface",
    "identifier": "uoft-cs/cifar10",
    "num_classes": 10,
    "constructor_args": {
        "input_key": "img",
        "target_key": "label"
    },
    "split_config": {
        "clean_train": {
            # Train on clean CIFAR-10
            "identifier": "uoft-cs/cifar10",
            "split_name": "train"
        },
        "clean_test": {
            # Test on clean CIFAR-10
            "identifier": "uoft-cs/cifar10", 
            "split_name": "test"
        },
        "corrupted_test": {
            # Test on corrupted/augmented version (if available)
            # Note: This would need a corrupted CIFAR-10 dataset on HF
            # For now using same dataset with different preprocessing to simulate corruption
            "identifier": "uoft-cs/cifar10",
            "split_name": "test",
            "preprocessing": corruption_preprocessing
        }
    }
}

print("\\n1. CIFAR-10 Clean vs Corrupted/Augmented:")
print("   ✅ Same classes: airplane, automobile, bird, cat, deer, dog, frog, horse, ship, truck")
print("   ✅ Same dimensions: 32x32x3")
print("   ✅ Same semantic meaning: class 0 = airplane in both datasets")
print("   🎯 Research question: How robust is the model to image corruption?")

# Example 2: Different datasets with overlapping semantic classes
print("\\n2. Real-world semantically compatible options (if available on HF):")
print("   ✅ CIFAR-10 ↔ STL-10 (subset): Both have overlapping object classes")
print("   ✅ CIFAR-10 ↔ ImageNet subset: Map to 10 overlapping classes")
print("   ✅ Fashion-MNIST ↔ Deep Fashion (subset): Both clothing categories")

# WARNING about incompatible combinations
print("\\n❌ AVOID these combinations (different semantic meanings):")
print("   ❌ CIFAR-10 (objects) ↔ MNIST (digits)")
print("   ❌ CIFAR-10 (objects) ↔ SVHN (digits)")
print("   ❌ Fashion-MNIST (clothing) ↔ MNIST (digits)")
print("   ❌ Any combination where class indices mean different things")

print("\\n🔑 KEY INSIGHT:")
print("Cross-domain evaluation only makes sense when:")
print("1. Same input dimensions (model compatibility)")
print("2. Same number of classes (output compatibility)")  
print("3. SAME SEMANTIC CLASS MEANINGS (evaluation validity)")
print("\\nClass 0 must mean the same thing in both datasets!")
print("Otherwise you're comparing apples to oranges!")

# Show the current configuration
print(f"\\nCurrent config constructor_args: {cifar_variants_params['constructor_args']}")
print("Configuration prioritizes semantic compatibility over dataset diversity.")
print("\\n✅ Using direct API parameters for maximum clarity and simplicity!")

\n=== SEMANTICALLY COMPATIBLE Cross-Domain Examples ===
\nCrucial requirement: Output classes must have SAME SEMANTIC MEANING!
Not just same number of classes - the classes themselves must correspond!
\n1. CIFAR-10 Clean vs Corrupted/Augmented:
   ✅ Same classes: airplane, automobile, bird, cat, deer, dog, frog, horse, ship, truck
   ✅ Same dimensions: 32x32x3
   ✅ Same semantic meaning: class 0 = airplane in both datasets
   🎯 Research question: How robust is the model to image corruption?
\n2. Real-world semantically compatible options (if available on HF):
   ✅ CIFAR-10 ↔ STL-10 (subset): Both have overlapping object classes
   ✅ CIFAR-10 ↔ ImageNet subset: Map to 10 overlapping classes
   ✅ Fashion-MNIST ↔ Deep Fashion (subset): Both clothing categories
\n❌ AVOID these combinations (different semantic meanings):
   ❌ CIFAR-10 (objects) ↔ MNIST (digits)
   ❌ CIFAR-10 (objects) ↔ SVHN (digits)
   ❌ Fashion-MNIST (clothing) ↔ MNIST (digits)
   ❌ Any combination where class indices mea

In [70]:
# Real-world examples of semantically compatible datasets on HuggingFace
print("\\n=== REAL HuggingFace Dataset Examples for Cross-Domain Evaluation ===")

# Example with actual compatible datasets (if they exist on HF)
print("\\n1. Object Classification (32x32, 10 classes):")
print("   Source: uoft-cs/cifar10")
print("   Target: STL-10 subset (if available) - overlapping classes")
print("   Classes: airplane, bird, car, cat, deer, dog, horse, ship, truck")
print("   Note: Would need to map/filter STL-10 to matching classes")

print("\\n2. Handwritten Digits (28x28, 10 classes):")
print("   Source: mnist")
print("   Target: emnist/digits subset")  
print("   Classes: 0, 1, 2, 3, 4, 5, 6, 7, 8, 9 (same digits)")
print("   Different handwriting styles but same semantic meaning")

print("\\n3. Fashion/Clothing (28x28, 10 classes):")
print("   Source: fashion_mnist")
print("   Target: fashion_mnist with different transformations")
print("   Classes: T-shirt, Trouser, Pullover, Dress, Coat, Sandal, Shirt, Sneaker, Bag, Ankle boot")

# Create a practical example with datasets that actually exist using direct API parameters
mnist_preprocessing = PreprocessConfig(
    steps=[
        PreprocessStep(name="Resize", params={"size": 28}),
        PreprocessStep(name="ToTensor"),
        PreprocessStep(name="ToDtype", params={"dtype": "torch.float32", "scale": True}),
        # Different normalization to simulate different image conditions
        PreprocessStep(
            name="Normalize",
            params={"mean": [0.3], "std": [0.4]}  # Different from standard [0.5], [0.5]
        ),
    ]
)

practical_params = {
    "dataset_name": "huggingface",
    "identifier": "ylecun/mnist",  # Using actual HF MNIST
    "num_classes": 10,
    "constructor_args": {
        "input_key": "image",  # MNIST typically uses "image"
        "target_key": "label"
    },
    "split_config": {
        "source_train": {
            "identifier": "ylecun/mnist",
            "split_name": "train"
        },
        "source_test": {
            "identifier": "ylecun/mnist",
            "split_name": "test"
        },
        # For now, using same dataset - in practice you'd use a compatible variant
        "target_test": {
            "identifier": "ylecun/mnist", 
            "split_name": "test",
            # Different preprocessing to simulate domain shift while maintaining semantics
            "preprocessing": mnist_preprocessing
        }
    }
}

print("\\n4. Practical Implementation Example:")
print("   Using MNIST with different preprocessing to simulate domain shift")
print("   Same digits (0-9) but different visual characteristics")
print(f"   Constructor args: {practical_params['constructor_args']}")

print("\\n💡 Pro Tips for Finding Compatible Datasets:")
print("1. Search HuggingFace for 'corrupted', 'augmented', or 'transformed' versions")
print("2. Look for datasets with explicit class mappings to standard benchmarks")
print("3. Check dataset papers for cross-dataset evaluation protocols")
print("4. Use dataset variants: CIFAR-10 → CIFAR-10-C → CIFAR-10.1")
print("5. Consider synthetic→real domain shifts with same object classes")

print("\\n⚠️  Always verify class mappings before cross-domain evaluation!")
print("The model's predictions are only meaningful if classes have same semantics.")
print("\\n✅ All examples use direct API parameters - cleaner than CLI config objects!")

\n=== REAL HuggingFace Dataset Examples for Cross-Domain Evaluation ===
\n1. Object Classification (32x32, 10 classes):
   Source: uoft-cs/cifar10
   Target: STL-10 subset (if available) - overlapping classes
   Classes: airplane, bird, car, cat, deer, dog, horse, ship, truck
   Note: Would need to map/filter STL-10 to matching classes
\n2. Handwritten Digits (28x28, 10 classes):
   Source: mnist
   Target: emnist/digits subset
   Classes: 0, 1, 2, 3, 4, 5, 6, 7, 8, 9 (same digits)
   Different handwriting styles but same semantic meaning
\n3. Fashion/Clothing (28x28, 10 classes):
   Source: fashion_mnist
   Target: fashion_mnist with different transformations
   Classes: T-shirt, Trouser, Pullover, Dress, Coat, Sandal, Shirt, Sneaker, Bag, Ankle boot
\n4. Practical Implementation Example:
   Using MNIST with different preprocessing to simulate domain shift
   Same digits (0-9) but different visual characteristics
   Constructor args: {'input_key': 'image', 'target_key': 'label'}
\n💡 P

## 8. Real-World Example: Semantically Compatible Datasets with split_config

### CIFAR-10 vs CIFAR-10-C - Perfect Semantic Compatibility

This example demonstrates using two different HuggingFace datasets that are **semantically identical**:
- **CIFAR-10**: Clean, high-quality images
- **CIFAR-10-C**: Same images with natural corruptions (noise, blur, weather effects, etc.)

Both datasets have:
- ✅ **Same dimensions**: 32×32 RGB images
- ✅ **Same number of classes**: 10 classes  
- ✅ **IDENTICAL semantic meanings**: 
  - Class 0 = airplane in BOTH datasets
  - Class 1 = automobile in BOTH datasets  
  - Class 2 = bird in BOTH datasets
  - ... and so on for all 10 classes

**✅ This is semantically VALID**: The model learns the same concepts and can be meaningfully evaluated across visual conditions!

In [71]:
# Example: SEMANTICALLY COMPATIBLE datasets using split_config
# CIFAR-10 (clean) vs CIFAR-10-C (corrupted) - SAME semantic classes!

print("=== Real-World Example: CIFAR-10 vs CIFAR-10-C ===")
print("✅ Semantically VALID - same classes, different visual conditions!")

# Preprocessing for 32x32 RGB images (works for both datasets)
cifar_preprocessing = PreprocessConfig(
    steps=[
        PreprocessStep(name="Resize", params={"size": 32}),
        PreprocessStep(name="CenterCrop", params={"size": 32}),
        PreprocessStep(name="ToTensor"),
        PreprocessStep(name="ToDtype", params={"dtype": "torch.float32", "scale": True}),
        PreprocessStep(name="Normalize", params={"mean": [0.485, 0.456, 0.406], "std": [0.229, 0.224, 0.225]})
    ]
)

# SEMANTICALLY COMPATIBLE datasets with IDENTICAL class meanings
cifar_robustness_params = {
    "dataset_name": "huggingface", 
    "identifier": "uoft-cs/cifar10",  # Default dataset (clean images)
    "num_classes": 10,
    "preprocessing": cifar_preprocessing,
    "constructor_args": {
        "input_key": "img",     # CIFAR-10 uses "img"
        "target_key": "label"   # Both use "label"
    },
    "split_config": {
        "clean_train": {
            # Train on clean CIFAR-10 images
            "identifier": "uoft-cs/cifar10",
            "split_name": "train"
        },
        "clean_test": {
            # Test on clean CIFAR-10 images
            "identifier": "uoft-cs/cifar10", 
            "split_name": "test"
        },
        "corrupted_test": {
            # Test on corrupted CIFAR-10-C images (SAME semantic classes!)
            "identifier": "randall-lab/cifar10-c",
            "split_name": "test",
            "constructor_args": {
                "input_key": "image",   # CIFAR-10-C uses "image"
                "target_key": "label"   # Both use "label"
            }
        }
    }
}

print("Dataset configuration:")
print(f"Default identifier: {cifar_robustness_params['identifier']}")
print(f"Number of classes: {cifar_robustness_params['num_classes']}")

print("\nSplit configuration:")
for split_name, config in cifar_robustness_params["split_config"].items():
    print(f"  {split_name}:")
    print(f"    Dataset: {config['identifier']}")
    print(f"    Split: {config['split_name']}")
    if 'constructor_args' in config:
        print(f"    Custom args: {config['constructor_args']}")

print("\n📊 Dataset Details:")
print("CIFAR-10 (uoft-cs/cifar10):")
print("  - 32×32 RGB images")
print("  - 10 classes: airplane, automobile, bird, cat, deer, dog, frog, horse, ship, truck")
print("  - Clean, high-quality images")

print("\nCIFAR-10-C (randall-lab/cifar10-c):")
print("  - 32×32 RGB images") 
print("  - SAME 10 classes: airplane, automobile, bird, cat, deer, dog, frog, horse, ship, truck")
print("  - 19 corruption types × 5 severity levels = 950k corrupted images")
print("  - Corruptions: Gaussian noise, motion blur, snow, fog, brightness, etc.")

print("\n✅ PERFECT Semantic Compatibility:")
print("  ✅ Class 0 = 'airplane' in BOTH datasets")
print("  ✅ Class 1 = 'automobile' in BOTH datasets") 
print("  ✅ Class 2 = 'bird' in BOTH datasets")
print("  ✅ ... all 10 classes have IDENTICAL meanings!")
print("  ✅ Model trained on clean images can be meaningfully tested on corrupted images")

print("\n🎯 Perfect Use Case for This Configuration:")
print("  ✅ Robustness evaluation: How well does model handle image corruption?")
print("  ✅ Domain adaptation: Clean → corrupted image performance")
print("  ✅ Adversarial robustness: Natural corruptions vs adversarial examples")
print("  ✅ Real-world deployment: Lab conditions → real-world conditions")

print("\n🔬 Research Questions This Enables:")
print("  • How robust is my model to natural image corruptions?")
print("  • Which corruption types are most challenging for my architecture?")
print("  • Does adversarial training improve natural corruption robustness?")

=== Real-World Example: CIFAR-10 vs CIFAR-10-C ===
✅ Semantically VALID - same classes, different visual conditions!
Dataset configuration:
Default identifier: uoft-cs/cifar10
Number of classes: 10

Split configuration:
  clean_train:
    Dataset: uoft-cs/cifar10
    Split: train
  clean_test:
    Dataset: uoft-cs/cifar10
    Split: test
  corrupted_test:
    Dataset: randall-lab/cifar10-c
    Split: test
    Custom args: {'input_key': 'image', 'target_key': 'label'}

📊 Dataset Details:
CIFAR-10 (uoft-cs/cifar10):
  - 32×32 RGB images
  - 10 classes: airplane, automobile, bird, cat, deer, dog, frog, horse, ship, truck
  - Clean, high-quality images

CIFAR-10-C (randall-lab/cifar10-c):
  - 32×32 RGB images
  - SAME 10 classes: airplane, automobile, bird, cat, deer, dog, frog, horse, ship, truck
  - 19 corruption types × 5 severity levels = 950k corrupted images
  - Corruptions: Gaussian noise, motion blur, snow, fog, brightness, etc.

✅ PERFECT Semantic Compatibility:
  ✅ Class 0 = 'air

In [ ]:
# Load the semantically compatible datasets to demonstrate split_config functionality
print("\n=== Loading Semantically Compatible Datasets ===")

try:
    # Load datasets using split_config with CIFAR-10 and CIFAR-10-C
    robustness_datasets = DatasetFactory.load_dataset(**cifar_robustness_params)
    
    print("✅ Successfully loaded semantically compatible datasets!")
    print(f"Available splits: {list(robustness_datasets.keys())}")
    
    # Show sample counts for each split
    for split_name, dataset in robustness_datasets.items():
        print(f"  {split_name}: {len(dataset)} samples")
    
    # Demonstrate accessing a sample from each split
    print("\n📋 Sample Data from Each Split:")
    
    for split_name, dataset in robustness_datasets.items():
        if len(dataset) > 0:
            sample = dataset[0]
            print(f"\n{split_name}:")
            print(f"  Image shape: {sample[0].shape}")
            print(f"  Label: {sample[1]} (class name depends on dataset)")
            print(f"  Label type: {type(sample[1])}")
    
    print("\n💡 Key Insights:")
    print("✅ The split_config successfully loaded data from TWO different datasets!")
    print("✅ clean_train & clean_test: From uoft-cs/cifar10 (clean images)")
    print("✅ corrupted_test: From randall-lab/cifar10-c (corrupted images)")
    print("✅ ALL datasets have IDENTICAL semantic class meanings!")
    print("✅ Class 0 = airplane in ALL three splits")
    print("✅ This enables meaningful robustness evaluation!")
    
    print("\n🔬 What This Enables:")
    print("- Train on clean images (clean_train)")
    print("- Validate on clean images (clean_test)")  
    print("- Test robustness on corrupted images (corrupted_test)")
    print("- Compare clean vs corrupted performance")
    print("- Measure model degradation under natural corruptions")
    
except Exception as e:
    print(f"❌ Error loading datasets: {e}")
    print("Note: CIFAR-10-C requires trust_remote_code=True and may need special handling")
    print("But the configuration syntax demonstrates perfect semantic compatibility!")


=== Loading Semantically Compatible Datasets ===


KeyboardInterrupt: 

## 7. Summary and Best Practices

### Key Takeaways:

1. **🎯 API vs CLI Usage Patterns**:
   - **API**: Use `DatasetFactory.load_dataset(**kwargs)` with direct parameters
   - **CLI**: Use `CreateDatasetCliConfig` objects in YAML files
   - **Don't mix**: Avoid creating CLI config objects in API code

2. **Split Priority System**:
   - `split_config` (highest) → `load_splits` (medium) → default `['train', 'test']` (lowest)
   - Use `load_splits` for simple cases, `split_config` for advanced scenarios

3. **Dataset Configuration Flexibility**:
   - Different datasets for different splits using `split_config`
   - Custom split name mappings
   - Split-specific preprocessing

4. **Hugging Face Integration**:
   - Use `constructor_args` with `input_key` and `target_key` for HF datasets
   - No redundant `dataset_kwargs` when `split_name` is specified
   - Seamless integration with existing AdvSecureNet workflows

5. **🔴 CRITICAL: Cross-Domain Semantic Compatibility**:
   - **Same input dimensions** (model architecture compatibility)
   - **Same number of classes** (output layer compatibility)
   - **SAME SEMANTIC MEANING for each class** (evaluation validity)
   - Class 0 must represent the same concept in both datasets!
   - ❌ Don't mix objects (CIFAR-10) with digits (MNIST/SVHN)
   - ✅ Use corrupted/augmented versions of same dataset
   - ✅ Use datasets with explicit class mappings

6. **Code Organization**:
   - API code: Direct parameters, no config objects
   - CLI code: Configuration objects and YAML files
   - Factory functions handle configuration resolution automatically

### Cross-Domain Evaluation Guidelines:
- **Valid**: CIFAR-10 (clean) → CIFAR-10-C (corrupted) 
- **Valid**: MNIST → EMNIST (same digits, different styles)
- **Invalid**: CIFAR-10 (objects) → SVHN (street numbers)
- **Invalid**: Fashion-MNIST (clothing) → MNIST (digits)

### API Best Practices:
- ✅ Use direct parameters with factories
- ✅ Define preprocessing configs once, reuse them
- ✅ Use split_config for complex scenarios
- ❌ Don't create CLI config objects in API code
- ❌ Don't use unnecessary intermediate configuration layers

# 📚 Comprehensive Guide: Global vs Split Arguments

This section provides a complete reference for understanding how arguments can be configured globally (affecting all splits) vs. per-split (split-specific overrides).

## 🎯 Key Concepts

### Global Arguments
- Set once at the top level
- Apply to **all splits** unless overridden
- Useful for consistent settings across train/validation/test

### Split Arguments  
- Set within individual split configurations
- **Override global settings** for that specific split
- Allow different datasets, preprocessing, or parameters per split

## 🔧 Configurable Arguments Reference

| Argument | Global Support | Split Support | Use Cases |
|----------|---------------|---------------|-----------|
| `identifier` | ✅ | ✅ | Same dataset for all splits vs. different datasets per split |
| `preprocessing` | ✅ | ✅ | Consistent preprocessing vs. split-specific transformations |
| `constructor_args` | ✅ | ✅ | Shared parameters vs. split-specific dataset arguments |
| `trust_remote_code` | ✅ | ✅ | Security settings - can vary by dataset source |
| `split_name` | ❌ | ✅ | Always split-specific (train/validation/test) |

## 📖 Practical Examples

### Example 1: Same Dataset, Different Splits

In [ ]:
# 🔄 Example 1: Same Dataset, Different Splits
# Global identifier - all splits use the same dataset

same_dataset_config = CreateDatasetCliConfig(
    # 🌍 GLOBAL SETTINGS - applied to all splits
    identifier="uoft-cs/cifar10",  # Same dataset for train/validation/test
    preprocessing=PreprocessConfig(
        steps=[
            PreprocessStep(name="Resize", params={"size": 32}),
            PreprocessStep(name="ToTensor"),
            PreprocessStep(name="Normalize", params={"mean": [0.5, 0.5, 0.5], "std": [0.5, 0.5, 0.5]})
        ]
    ),
    constructor_args={"trust_remote_code": True},
    
    # 📋 SPLIT-SPECIFIC SETTINGS - only split_name varies
    split_config={
        "train": {"split_name": "train"},      # Uses global identifier: uoft-cs/cifar10
        "validation": {"split_name": "test"},  # Uses global identifier: uoft-cs/cifar10  
        "test": {"split_name": "test"}         # Uses global identifier: uoft-cs/cifar10
    }
)

print("🔄 Same Dataset Configuration:")
print(f"✅ All splits use: {same_dataset_config.identifier}")
print(f"✅ Shared preprocessing: {same_dataset_config.preprocessing}")
print(f"✅ Train split: train split of uoft-cs/cifar10")
print(f"✅ Validation split: test split of uoft-cs/cifar10") 
print(f"✅ Test split: test split of uoft-cs/cifar10")

TypeError: UserSplitConfig.__init__() got an unexpected keyword argument 'train'

In [ ]:
# 🎯 Example 2: Different Datasets Per Split  
# No global identifier - each split specifies its own dataset

different_datasets_config = CreateDatasetCliConfig(
    # 🌍 GLOBAL SETTINGS - shared across splits where not overridden
    preprocessing=PreprocessConfig(
        steps=[
            PreprocessStep(name="Resize", params={"size": 32}),
            PreprocessStep(name="ToTensor"),
            PreprocessStep(name="Normalize", params={"mean": [0.5, 0.5, 0.5], "std": [0.5, 0.5, 0.5]})
        ]
    ),
    # Note: No global identifier - each split defines its own
    
    # 📋 SPLIT-SPECIFIC SETTINGS - each split uses different dataset
    split_config={
        "train": {
            "identifier": "uoft-cs/cifar10",           # Clean CIFAR-10 for training
            "split_name": "train",
            "constructor_args": {"trust_remote_code": True}
        },
        "validation": {
            "identifier": "uoft-cs/cifar10",           # Clean CIFAR-10 for validation
            "split_name": "test", 
            "constructor_args": {"trust_remote_code": True}
        },
        "test": {
            "identifier": "randall-lab/cifar10-c",     # Corrupted CIFAR-10-C for testing
            "split_name": "test",
            "constructor_args": {"trust_remote_code": True}
        }
    }
)

print("🎯 Different Datasets Configuration:")
print(f"✅ Train uses: uoft-cs/cifar10 (clean data)")
print(f"✅ Validation uses: uoft-cs/cifar10 (clean data)")  
print(f"✅ Test uses: randall-lab/cifar10-c (corrupted data)")
print(f"✅ All splits share: same preprocessing pipeline")
print(f"✅ All splits share: trust_remote_code=True")

In [ ]:
# ⚖️ Example 3: Mixed Global and Split-Specific Settings
# Demonstrates global settings with split-specific overrides

mixed_settings_config = CreateDatasetCliConfig(
    # 🌍 GLOBAL SETTINGS - defaults for all splits
    identifier="uoft-cs/cifar10",  # Default dataset
    preprocessing=PreprocessConfig(
        steps=[
            PreprocessStep(name="Resize", params={"size": 32}),
            PreprocessStep(name="ToTensor"),
            PreprocessStep(name="Normalize", params={"mean": [0.5, 0.5, 0.5], "std": [0.5, 0.5, 0.5]})
        ]
    ),
    constructor_args={"trust_remote_code": True},  # Default security setting
    
    # 📋 SPLIT-SPECIFIC OVERRIDES
    split_config={
        "train": {
            "split_name": "train",
            # Uses global identifier: uoft-cs/cifar10
            # Uses global preprocessing
            # Uses global constructor_args
        },
        "validation": {
            "split_name": "test",
            # Overrides global preprocessing for validation-specific augmentation
            "preprocessing": PreprocessConfig(
                steps=[
                    PreprocessStep(name="Resize", params={"size": 32}),
                    PreprocessStep(name="ToTensor"),
                    PreprocessStep(name="Normalize", params={"mean": [0.5, 0.5, 0.5], "std": [0.5, 0.5, 0.5]}),
                    PreprocessStep(name="ToDtype", params={"dtype": "torch.float32", "scale": True})  # Additional validation-specific setting
                ]
            )
        },
        "test": {
            # Completely overrides global settings for test split
            "identifier": "randall-lab/cifar10-c",  # Different dataset!
            "split_name": "test",
            "constructor_args": {"trust_remote_code": True},  # Same as global
            "preprocessing": PreprocessConfig(
                steps=[
                    PreprocessStep(name="Resize", params={"size": 32}),
                    PreprocessStep(name="ToTensor"),
                    PreprocessStep(name="Normalize", params={"mean": [0.485, 0.456, 0.406], "std": [0.229, 0.224, 0.225]})  # ImageNet normalization
                ]
            )
        }
    }
)

print("⚖️ Mixed Settings Configuration:")
print("📍 Train Split:")
print("  • Dataset: uoft-cs/cifar10 (from global)")
print("  • Preprocessing: standard normalization (from global)")
print("  • Constructor args: trust_remote_code=True (from global)")

print("\n📍 Validation Split:")  
print("  • Dataset: uoft-cs/cifar10 (from global)")
print("  • Preprocessing: standard + to_tensor=True (OVERRIDE)")
print("  • Constructor args: trust_remote_code=True (from global)")

print("\n📍 Test Split:")
print("  • Dataset: randall-lab/cifar10-c (OVERRIDE)")
print("  • Preprocessing: ImageNet normalization (OVERRIDE)")  
print("  • Constructor args: trust_remote_code=True (explicit)")

## 🚀 Advanced Patterns

### Pattern 1: Cross-Domain Evaluation
```python
# Train on clean data, test on corrupted data
cross_domain_config = CreateDatasetCliConfig(
    # Global preprocessing for consistency
    preprocessing=PreprocessConfig(
        steps=[
            PreprocessStep(name="Resize", params={"size": 32}),
            PreprocessStep(name="ToTensor"),
            PreprocessStep(name="Normalize", params={"mean": [0.5, 0.5, 0.5], "std": [0.5, 0.5, 0.5]})
        ]
    ),
    
    split_config={
        "train": {"identifier": "uoft-cs/cifar10", "split_name": "train"},
        "test": {"identifier": "randall-lab/cifar10-c", "split_name": "test"}  # Different domain!
    }
)
```

### Pattern 2: Split-Specific Constructor Args  
```python
# Different constructor arguments per dataset
split_constructor_config = CreateDatasetCliConfig(
    preprocessing=PreprocessConfig(
        steps=[
            PreprocessStep(name="Resize", params={"size": 224}),
            PreprocessStep(name="ToTensor")
        ]
    ),  # Global preprocessing
    
    split_config={
        "train": {
            "identifier": "dataset-a",
            "split_name": "train", 
            "constructor_args": {"field_mapping": {"image": "img", "label": "target"}}
        },
        "test": {
            "identifier": "dataset-b", 
            "split_name": "test",
            "constructor_args": {"field_mapping": {"picture": "img", "class": "target"}}
        }
    }
)
```

### Pattern 3: Security-Conscious Configuration
```python
# Different trust levels per dataset source
security_config = CreateDatasetCliConfig(
    preprocessing=PreprocessConfig(
        steps=[
            PreprocessStep(name="Resize", params={"size": 32}),
            PreprocessStep(name="ToTensor")
        ]
    ),  # Shared preprocessing
    
    split_config={
        "train": {
            "identifier": "trusted-org/dataset",
            "split_name": "train",
            "constructor_args": {"trust_remote_code": True}   # Trust official source
        },
        "test": {
            "identifier": "community/dataset", 
            "split_name": "test",
            "constructor_args": {"trust_remote_code": False}  # Be cautious with community data
        }
    }
)
```

## 🎯 Decision Matrix: When to Use Global vs Split Settings

| Scenario | Recommended Approach | Rationale |
|----------|---------------------|-----------|
| **Same dataset, different splits** | Global `identifier` | Reduces duplication, ensures consistency |
| **Cross-domain evaluation** | Split-specific `identifier` | Each split needs different dataset |
| **Consistent preprocessing** | Global `preprocessing` | Same transformations across all splits |
| **Split-specific augmentation** | Split-specific `preprocessing` | Different augmentations for train vs test |
| **Mixed dataset sources** | Split-specific `constructor_args` | Different datasets may need different parameters |
| **Security requirements** | Split-specific `trust_remote_code` | Trust levels may vary by data source |

## ⚡ Best Practices

### ✅ DO
- Use **global settings** for consistent parameters across splits
- Use **split-specific overrides** when you need different behavior per split  
- Be **explicit** in split configs when using multiple datasets
- Document your **reasoning** for global vs split choices

### ❌ DON'T
- Mix global and split `identifier` unnecessarily (choose one pattern)
- Duplicate identical settings across splits (use global instead)
- Forget to set `trust_remote_code` when required by the dataset
- Assume global settings will work for all datasets (some need split-specific args)

## 🔍 Debugging Tips

### Common Issues and Solutions

1. **"Dataset not found" errors**
   - Check if `identifier` is set globally or per split
   - Verify `trust_remote_code` is set when required

2. **Field mapping errors** 
   - Different datasets may use different field names
   - Use split-specific `constructor_args` for field mapping

3. **Preprocessing conflicts**
   - Global preprocessing may not suit all datasets  
   - Override with split-specific preprocessing when needed

4. **Inconsistent results across splits**
   - Verify preprocessing is consistent where intended
   - Check if split overrides are unintentional